In [ ]:
!pip -q install webdataset huggingface_hub

In [ ]:
pip install torch-lucent

In [ ]:
import torch
import numpy as np
import heapq
from PIL import Image
import matplotlib.pyplot as plt
import os, re
import webdataset as wds
from lucent.modelzoo import inceptionv1
from lucent.optvis import render, param, transform, objectives
import pickle as pkl

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#Enter your HuggingFace Token here
HF_TOKEN = "_"
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN len:", len(os.environ["HF_TOKEN"]))

In [ ]:
import os, webdataset as wds
from torch.utils.data import DataLoader
from torchvision import transforms

# preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def to_tensor(sample):
    img = sample.get("jpg") or sample.get("jpeg") or sample.get("JPEG")
    if img is None:
        return None
    key = sample.get("__key__", "")
    return preprocess(img), key

base = "https://huggingface.co/datasets/timm/imagenet-1k-wds/resolve/main/"
train_shards = [f"{base}imagenet1k-train-{i:04d}.tar?download=true" for i in range(1024)]

URLS_TRAIN = [
    "pipe:curl -f -sSL --retry 20 --retry-delay 2 "
    "-H \"Authorization: Bearer $HF_TOKEN\" "
    f"\"{u}\""
    for u in train_shards
]

dataset_train = (
    wds.WebDataset(URLS_TRAIN, handler=wds.handlers.warn_and_continue)
      .decode("pil", handler=wds.handlers.warn_and_continue)
      .map(to_tensor, handler=wds.handlers.warn_and_continue)
      .select(lambda x: x is not None)
)

dl_train = DataLoader(dataset_train, batch_size=128, num_workers=0, pin_memory=False)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = inceptionv1(pretrained=True).to(device).eval()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Configurations
# Kindly change the LAYER_NAME and CHANNEL_START, CHANNEL_END according to your respective responsibility

CFG = {
    "LAYER_NAME": "mixed4e",
    "CHANNEL_START": 0,
    "CHANNEL_END": 832,
    "TOPK": 10,
    "HEAP_BATCH": 20,
    "REDUCTION": "max",
    "OUT_ROOT" : "/content/drive/MyDrive/Dataset_images",
    "CROP_FRAC": 0.25,
}

In [ ]:
# Image helpers
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def tensor_to_pil(x: torch.Tensor) -> Image.Image:
    x = x.detach().cpu()
    if x.dim() == 4:
        x = x[0]
    x = (x * std + mean).clamp(0, 1)
    x = (x.permute(1,2,0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(x)

def crop_from_tensor_and_feat(img_tensor_norm: torch.Tensor,
                              feat_single: torch.Tensor,
                              channel_id: int,
                              frac: float = 0.25) -> Image.Image:
    img = (img_tensor_norm.detach().cpu() * std + mean).clamp(0, 1)
    img_np = (img.permute(1,2,0).numpy() * 255).astype(np.uint8)
    H, W = img_np.shape[:2]

    act = feat_single[channel_id].detach().cpu().clamp(min=0)  # [h,w]
    h, w = act.shape

    flat = int(torch.argmax(act).item())
    iy, ix = flat // w, flat % w

    cy = int((iy + 0.5) * H / h)
    cx = int((ix + 0.5) * W / w)

    ch, cw = int(H * frac), int(W * frac)
    y0 = max(0, cy - ch // 2); y1 = min(H, y0 + ch)
    x0 = max(0, cx - cw // 2); x1 = min(W, x0 + cw)

    return Image.fromarray(img_np[y0:y1, x0:x1])


In [ ]:
# Core scoring helper

@torch.no_grad()
def reduce_all_channels(feat: torch.Tensor, reduction: str) -> torch.Tensor:
    # feat: [B,C,h,w] -> [B,C]
    if reduction == "max":
        return feat.amax(dim=(2,3))
    elif reduction == "mean":
        return feat.mean(dim=(2,3))
    else:
      raise ValueError("reduction must be 'max' or 'mean'")

In [ ]:
# Pass 1: scan dataset once

CKPT_PATH = os.path.join(
    CFG["OUT_ROOT"],
    f"{CFG['LAYER_NAME']}_topk_scan_ckpt.pkl"
)
SAVE_EVERY = 50

def save_ckpt(path, step, heaps, ch0, ch1, layer_name):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    tmp_path = path + ".tmp"

    checkpoint = {
        "step": step,
        "heaps": heaps,
        "ch0": ch0,
        "ch1": ch1,
        "layer_name": layer_name,
        "channel_start": CFG["CHANNEL_START"],
        "channel_end": CFG["CHANNEL_END"],
        "topk": CFG["TOPK"],
        "batch_size": dl_train.batch_size,
    }

    with open(tmp_path, "wb") as f:
        pkl.dump(checkpoint, f)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def load_ckpt(path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pkl.load(f)
    return None

def scan_topk_keys(model, dl_train, device, layer_name: str,
                   channel_start: int, channel_end_inclusive: int,
                   topk: int, heap_batch: int, reduction: str):
    activation = {}

    def hook_fn(module, inp, out):
        activation["feat"] = out

    layer = dict(model.named_modules())[layer_name]
    handle = layer.register_forward_hook(hook_fn)

    model.eval()

    heaps = None
    ch0 = None
    ch1 = None
    start_step = 0

    # load checkpoint if exists
    ckpt = load_ckpt(CKPT_PATH)

    if ckpt is not None:
        same_run = (
            ckpt.get("layer_name") == layer_name
            and ckpt.get("channel_start") == channel_start
            and ckpt.get("channel_end") == channel_end_inclusive
            and ckpt.get("topk") == topk
            and ckpt.get("batch_size") == dl_train.batch_size
        )

        if same_run:
            start_step = int(ckpt["step"]) + 1
            heaps = ckpt["heaps"]
            ch0 = int(ckpt["ch0"])
            ch1 = int(ckpt["ch1"])

            print(
                f"Checkpoint found. Resuming after batch "
                f"{ckpt['step']}."
            )
        else:
            print("Checkpoint belongs to a different configuration. Starting fresh.")

    last_step = start_step - 1

    try:
        with torch.no_grad():
            for step, (imgs, keys) in enumerate(dl_train):

                # skip already-processed batches when resuming
                if step < start_step:
                    continue

                last_step = step

                imgs = imgs.to(device, non_blocking=True)

                use_cuda = imgs.is_cuda
                with torch.amp.autocast(device_type="cuda", enabled=use_cuda):
                    _ = model(imgs)

                feat = activation["feat"]  # [B,C,h,w]
                B, C = feat.shape[0], feat.shape[1]

                if heaps is None:
                    # clamp channel range to available channels
                    ch0 = max(0, int(channel_start))
                    ch1 = min(C - 1, int(channel_end_inclusive))
                    if ch0 > ch1:
                        raise ValueError(f"Invalid channel range after clamping: [{ch0}, {ch1}] with C={C}")

                    heaps = {c: [] for c in range(ch0, ch1 + 1)}
                    print(f"Scanning layer={layer_name} channels={C} using range [{ch0}, {ch1}]")

                else:
                    # sanity clamp if resuming from ckpt created with same layer
                    ch0 = max(0, min(ch0, C - 1))
                    ch1 = max(0, min(ch1, C - 1))
                    if ch0 > ch1:
                        raise ValueError(f"Checkpoint channel range invalid for this run: [{ch0}, {ch1}] with C={C}")

                scores_bc = reduce_all_channels(feat, reduction).float()  # [B,C]
                scores_sel = scores_bc[:, ch0:ch1+1]                      # [B,Cr]

                k = min(heap_batch, B)
                vals, idxs = torch.topk(scores_sel, k=k, dim=0)           # [k,Cr]

                vals = vals.detach().cpu()
                idxs = idxs.detach().cpu()

                # update heaps
                for offset, c in enumerate(range(ch0, ch1 + 1)):
                    h = heaps[c]
                    for j in range(k):
                        v = float(vals[j, offset].item())
                        i = int(idxs[j, offset].item())
                        key = keys[i]

                        item = (v, key)
                        if len(h) < topk:
                            heapq.heappush(h, item)
                        else:
                            if v > h[0][0]:
                                heapq.heapreplace(h, item)

                if (step + 1) % 50 == 0:
                    sample_c = ch0
                    if heaps[sample_c]:
                        print(f"Scanning batches={step+1} ch{sample_c} min_top{topk}={heaps[sample_c][0][0]:.4f}")

                # periodic checkpoint save
                if (step + 1) % SAVE_EVERY == 0:
                    save_ckpt(CKPT_PATH, step, heaps, ch0, ch1, layer_name)
                    print(f"Checkpoint saved after batch {step}")

    except KeyboardInterrupt:
        print("\nRun interrupted manually.")

    finally:
        if heaps is not None and last_step >= start_step:
            save_ckpt(
                CKPT_PATH,
                last_step,
                heaps,
                ch0,
                ch1,
                layer_name,
            )
            print(f"Latest state saved after batch {last_step}")

        handle.remove()

    # finalize sorted lists
    topk_by_channel = {c: sorted(h, key=lambda x: x[0], reverse=True) for c, h in heaps.items()}
    return topk_by_channel


In [ ]:
# Pass 2: save images to disk/Drive

def map_top_keys(topk_by_channel):
    wanted_keys = set()
    key_to_targets = {}  # key -> list[(channel, score, rank)]

    for c, items in topk_by_channel.items():
        for rank, (score, key) in enumerate(items, start=1):
            wanted_keys.add(key)
            key_to_targets.setdefault(key, []).append((c, float(score), rank))

    return wanted_keys, key_to_targets

def _safe_filename(s: str, max_len: int = 120) -> str:
    s = str(s).replace("/", "_").replace("\\", "_")
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", s)
    return s[:max_len]

@torch.no_grad()

def save_topk_images_wds_streaming(
    model, dl_train, device,
    layer_name: str,
    topk_by_channel: dict,
    out_root: str,
    crop_frac: float,
):
    # Build wanted maps
    wanted_keys, key_to_targets = map_top_keys(topk_by_channel)

    layer_dir = os.path.join(out_root, layer_name)
    os.makedirs(layer_dir, exist_ok=True)

    remaining = set()

    for key in wanted_keys:
        targets = key_to_targets[key]
        key_safe = _safe_filename(key)

        key_complete = True

        for c, score, rank in targets:
            ch_dir = os.path.join(layer_dir, f"ch_{c:04d}")

            full_path = os.path.join(
                ch_dir,
                f"rank{rank:02d}_FULL_score{score:.4f}_{key_safe}.jpg",
            )
            crop_path = os.path.join(
                ch_dir,
                f"rank{rank:02d}_CROP_score{score:.4f}_{key_safe}.jpg",
            )

            if not os.path.exists(full_path) or not os.path.exists(crop_path):
                key_complete = False
                break

        if not key_complete:
            remaining.add(key)

    print(
        f"Pass 2 resume: {len(wanted_keys) - len(remaining)} samples "
        f"already complete; {len(remaining)} still needed."
    )

    if not remaining:
        print("All top-k images are already saved.")
        return

    # Hook
    activation = {}
    def hook_fn(module, inp, out):
        activation["feat"] = out

    layer = dict(model.named_modules())[layer_name]
    handle = layer.register_forward_hook(hook_fn)

    model.eval()

    for step, (imgs, keys) in enumerate(dl_train):
        imgs = imgs.to(device, non_blocking=True)

        use_cuda = imgs.is_cuda
        with torch.amp.autocast(device_type="cuda", enabled=use_cuda):
            _ = model(imgs)

        feat = activation["feat"]  # [B,C,h,w]

        # Search the indices we need in this batch
        hit_indices = [i for i, k in enumerate(keys) if k in remaining]
        if not hit_indices:
            if (step + 1) % 200 == 0:
                print(f"batches={step+1} remaining={len(remaining)}")
            continue

        for i in hit_indices:
            key = keys[i]
            key_safe = _safe_filename(key)

            # produce PIL images once per sample
            pil_full = tensor_to_pil(imgs[i])
            # crop depends on channel, so computing per channel

            # save for all channels that want this key
            for (c, score, rank) in key_to_targets[key]:
                ch_dir = os.path.join(layer_dir, f"ch_{c:04d}")
                os.makedirs(ch_dir, exist_ok=True)

                pil_crop = crop_from_tensor_and_feat(imgs[i], feat[i], c, frac=crop_frac)

                pil_full.save(os.path.join(ch_dir, f"rank{rank:02d}_FULL_score{score:.4f}_{key_safe}.jpg"))
                pil_crop.save(os.path.join(ch_dir, f"rank{rank:02d}_CROP_score{score:.4f}_{key_safe}.jpg"))

            remaining.remove(key)

        if (step + 1) % 50 == 0:
            print(f"batches={step+1} saved={(len(wanted_keys)-len(remaining))}/{len(wanted_keys)} remaining={len(remaining)}")

        # Early exit when done
        if not remaining:
            print(f"Done early at batch {step+1}")
            break

    handle.remove()
    print("Done ->", layer_dir)


In [ ]:
# Run

topk_by_channel = scan_topk_keys(
    model=model,
    dl_train=dl_train,
    device=device,
    layer_name=CFG["LAYER_NAME"],
    channel_start=CFG["CHANNEL_START"],
    channel_end_inclusive=CFG["CHANNEL_END"],
    topk=CFG["TOPK"],
    heap_batch=CFG["HEAP_BATCH"],
    reduction=CFG["REDUCTION"],
)

save_topk_images_wds_streaming(
    model=model,
    dl_train=dl_train,
    device=device,
    layer_name=CFG["LAYER_NAME"],
    topk_by_channel=topk_by_channel,
    out_root=CFG["OUT_ROOT"],
    crop_frac=CFG["CROP_FRAC"],
)